## 06 JOINs: cruzar tablas para responder preguntas complejas

INNER JOIN y LEFT JOIN — combinar datos de varias tablas en una sola consulta.



1. Hasta ahora has trabajado con una tabla a la vez. 
2. Los datos reales están NORMALIZADOS: 
    - la información del cliente está en una tabla, sus pedidos en otra, los productos en otra. 
3. ¿Por qué? 
    - Porque repetir "Ana García, Madrid, ana@email.com" en cada uno de sus 50 pedidos es desperdiciar espacio y crear inconsistencias. 
4. En vez de eso, cada pedido guarda solo el cliente_id y la tabla clientes tiene el detalle.



Los JOINs son la forma de volver a UNIR esas tablas separadas cuando necesitas responder preguntas que cruzan datos: 

- "¿Qué clientes de Madrid han comprado productos de Electrónica?" 
  - requiere cruzar clientes + pedidos + productos. 

Es como juntar dos hojas de Excel que comparten una columna en común.



## Nuestro esquema: las tablas que vas a cruzar


**Consejo de senior:**

1. cuando llegas a un proyecto nuevo, lo primero que pido es el diagrama ER. 
2. Sin él estás adivinando qué tablas existen y cómo se conectan. 
3. Guarda este diagrama a mano
   - te va a servir para todos los ejercicios de esta lección y las siguientes.



## La analogía de las dos listas


**INNER JOIN vs LEFT JOIN**

1. INNER JOIN:
    - Solo filas que coinciden en AMBAS tablas
      - Clientes sin pedidos: NO aparecen
      - Pedidos sin cliente: NO aparece
2. LEFT JOIN:
    - TODAS las filas de la tabla izquierda + coincidencias
      - Clientes sin pedidos: Sí aparecen (con NULLs)
      - Pedido sin cliente: NO aparece
3. Cuándo usar cada uno?
    - INNER JOIN:
      - "CLientes que HAN comprado?" (excluye los que no)
    - LEFT JOIN:
      - "TODOS los clientes, compren o no" (incluye todos)

- **Regla:** si necesitas ver los que NO coinciden, usa LEFT JOIN
- INNER JOIN excluye lo que no coincide
- LEFT JOIN preserva todo el lado izquierdo

## INNER JOIN: solo las coincidencias


1. INNER JOIN (o simplemente JOIN) devuelve SOLO las filas donde hay coincidencia en ambas tablas. 
    - Si un cliente no tiene pedidos, desaparece. 
    - Si un pedido referencia un cliente que no existe, también desaparece. 
2. Mucha gente lo explica como "la intersección del diagrama de Venn" — y hasta cierto punto vale. 
    - Pero hay una trampa que te pillará tarde o temprano: un JOIN puede MULTIPLICAR filas. 
    - Si un cliente tiene 10 pedidos, el resultado tendrá 10 filas para ese cliente. 
    - No es un filtro, es un cruce. 
    - La primera vez que me pasó, pensé que la query estaba mal. No lo estaba. Es así como funciona.



```sql
-- Sintaxis del INNER JOIN
SELECT c.nombre, c.ciudad, p.importe, p.fecha
FROM clientes c
INNER JOIN pedidos p ON p.clientes_id  = c.id
ORDER BY p.fecha DEC
LIMIT 10;

-- "JOIN" sin prefijo es lo mismo que "INNER JOIN"
SELECT c.nombre, p.importe
FROM clientes c
JOIN pedidos p ON p.clientes_id = c.id
LIMIT 5;
```

**INNER JOIN: la cláusula ON define la condición de unión**

- FROM clientes c: tabla izquierda con alias "c" (abreviatura para no escribir "clientes." cada vez).
- INNER JOIN pedidos p: tabla derecha con alias "p".
- ON p.cliente_id = c.id: la condición de unión: vincula cada pedido con su cliente mediante la FK.
- c.nombre, c.ciudad: columnas de clientes. p.importe, p.fecha → columnas de pedidos.

"JOIN" sin prefijo es equivalente a "INNER JOIN". Solo devuelve filas con coincidencia en AMBAS tablas (clientes sin pedidos no aparecen).

## La anatomía de un JOIN

- FROM tabla_izquierda alias — la primera tabla
- JOIN tabla_derecha alias — la segunda tabla que quieres cruzar
- ON condicion — cómo se relacionan (normalmente FK = PK)
- Los alias (c, p) son esenciales para indicar de qué tabla viene cada columna
- Puedes encadenar múltiples JOINs: FROM a JOIN b ON... JOIN c ON...

## LEFT JOIN: preservar la tabla izquierda


1. LEFT JOIN devuelve TODAS las filas de la tabla izquierda (la del FROM), y las coincidencias de la derecha. 
2. Si no hay coincidencia, las columnas de la tabla derecha vienen como NULL. 
3. Esto es crucial para preguntas como "¿qué clientes NO han comprado nunca?".


```sql
-- LEFT JOIN: todos los clientes, incluso sin pedidos
SELECT c.nombre, c.ciudad, COUNT(p.id) AS pedidos
FROM clientes c
LEFT JOIN pedidos p ON p.clientes_id = c.id
GROUP BY c.id, c.nombre, c.ciudad
ORDER BY pedidos ASC
LIMIT 10;

-- Encontrar clientes que NUNCA han comprado
SELECT c.nombre, c.email, c.ciudad
FROM clientes c
LEFT JOIN pedidos p ON p.cliente_id = c.id
WHERE p.id IS NULL;
```

**LEFT JOIN + WHERE ... IS NULL = encontrar los que NO coinciden**


- Primera query: LEFT JOIN + GROUP BY: 
    - muestra TODOS los clientes con su COUNT de pedidos. 
    - Los que no tienen pedidos muestran 0 (COUNT de NULLs es 0). 
    - ORDER BY pedidos ASC pone los inactivos primero.
- Segunda query: LEFT JOIN + WHERE p.id IS NULL: 
    - patrón ANTI-JOIN. 
    - El LEFT JOIN mantiene todos los clientes; para los sin pedidos, p.id es NULL. 
    - El WHERE filtra SOLO esos. 
    - Resultado: clientes que NUNCA han comprado.

Este segundo patrón (LEFT JOIN + IS NULL) es equivalente a "NOT IN" o "NOT EXISTS" pero suele ser más legible y a veces más eficiente.



El patrón "LEFT JOIN + WHERE columna_derecha IS NULL" es un anti-join: 
1. encuentra lo que NO existe en la otra tabla. Es el equivalente de "buscar lo que falta". 
2. Úsalo para encontrar clientes sin pedidos, productos sin ventas, empleados sin departamento...



## JOINs con agregación: el combo más poderoso


El verdadero poder aparece cuando combinas JOINs con GROUP BY: 
- "gasto total por ciudad" requiere JOIN (para tener la ciudad del cliente) + SUM + GROUP BY. 

Este combo es probablemente la consulta más frecuente en cualquier empresa:



```sql
-- Ventas totales por ciudad (JOIN + GROUP BY)
SELECT
    c.ciudad,
    COUNT(p.id) AS pedidos,
    ROUND(SUM(p.importe), 2) AS total,
    ROUND(AVG(p.importe), 2) AS ticket_medio
FROM clientes c
JOIN pedidos p ON p.cliente_id = c.id
WHERE p.estado = 'completado'
GROUP BY c.ciudad
ORDER BY total DESC;

-- Categoría más vendida (JOIN + GROUP BY con categorías)
SELECT
    cat.nombre AS categoria,
    COUNT(p.id) AS pedidos,
    ROUND(SUM(p.importe), 2) AS total
FROM pedidos p
JOIN products pr ON pr.id = p.producto_id
JOIN categorias cat ON cat.id = pr.categoria_id
WHERE p.estado = 'completado'
GROUP BY cat.nombre
ORDER BY total DESC;
```

**JOIN + GROUP BY: cruzar datos de varias tablas y luego agregar**

El combo más frecuente en SQL empresarial:

1. Primera query: 
     - métricas de ventas POR CIUDAD. 
   - JOIN clientes + pedidos > GROUP BY c.ciudad:
   - La ciudad viene de clientes, el importe de pedidos. 
   - Sin JOIN no podrías cruzarlos.
2. Segunda query: 
     -  métricas POR CATEGORÍA. 
   - JOIN pedidos + productos + categorias > GROUP BY cat.nombre:
   -  La categoría vive en su propia tabla (normalizado), así que necesitas dos JOINs para llegar a ella.

**Flujo lógico:**

- FROM + JOIN > WHERE (filtra filas individuales) > GROUP BY (agrupa) > funciones de agregación > ORDER BY.

**Nota:**
- WHERE p.estado = 'completado' se aplica ANTES del GROUP BY, así que solo pedidos completados entran en la agregación.




1. Cuando haces JOIN y luego COUNT(*), cuidado: si un cliente tiene 3 pedidos, aparece 3 veces ANTES del GROUP BY. 
2. COUNT(*) contará esas 3 filas. Si quieres contar clientes únicos, usa COUNT(DISTINCT c.id).


## Errores comunes en JOINs


1. Olvidar el ON: sin condición de unión, obtienes un CROSS JOIN (producto cartesiano — millones de filas basura)
2. Condición ON incorrecta: si pones ON p.id = c.id en vez de ON p.cliente_id = c.id, cruzas por los IDs equivocados
3. No usar alias: sin alias, DuckDB no sabe si "id" viene de clientes o de pedidos
4. LEFT JOIN pero filtrar la tabla derecha en WHERE: WHERE p.estado = 'completado' ANULA el LEFT JOIN (convierte NULLs en no-coincidencia)

**¿Y RIGHT JOIN?**

- Existe, pero en la práctica todo RIGHT JOIN se puede reescribir como LEFT JOIN invirtiendo el orden de las tablas. 
- Por eso casi nadie lo usa: es más legible poner la tabla "principal" a la izquierda y hacer LEFT JOIN con la secundaria.
-  Si lo ves en código legacy, ya sabes qué hace pero no lo necesitas.


